# NQS vs exact perturbation energies — $h_x=0$ cut

$E_{\rm NQS}$ against the exact low-field series on the pure $h_z$ cut, and the relative error
$|E_{\rm NQS}-E_{\rm pert}|/|E_{\rm NQS}|$. Series coefficients come from
`analysis/exact_benchmarks.py` (derived from this repo's own geometry, **zero fitted parameters,
zero NQS input**):

$$E = -(\#A_v+\#B_p) - c_2h_z^2 - c_4h_z^4 + O(h_z^6),\quad
c_2=\sum_i\tfrac{1}{2n_v(i)},\quad c_4=\tfrac{5}{16}\#B_p+\tfrac{N_{\rm adj}}{32}-\tfrac{N}{64}$$

Two columns are not optional. **`E_err`**: $E_{\rm NQS}$ is a *single* `vs.expect(Ham)` on the
final state (`validation.py:169`), not a step average, so it has an MC error of the mean — the
`sigma` column says whether the deviation is resolved at all. **`c6~`**: a rough $0.45Nh_z^6$
scale for the first neglected term; once it approaches the deviation, the comparison is measuring
series truncation rather than NQS error.

In [ ]:
# ====================== 1 · CONFIG — the one cell to edit ======================
import json, glob, os, sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "analysis"
                else os.getcwd())
from analysis.exact_benchmarks import counts, E_lowfield

# ---- global plot style (spec) ----
plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False,
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
})

ROOT       = "/Users/sanzhar123/Desktop/Approximate-Symmetries-TC-main/results"
ENERGY_DIR = f"{ROOT}/phase_hx0.0_energy"                 # aggregated E(hz) per L
CURVE_DIR  = os.path.join(os.path.dirname(ROOT), "data/tc_nqs/phase_hx0.0")  # per-run curves
FIGDIR     = os.path.join(os.path.dirname(ROOT), "figures")   # gitignored; savefig target
os.makedirs(FIGDIR, exist_ok=True)

LS      = [4, 5, 6, 7]      # system sizes (present ones auto-filter)
BC      = "OBC"
HMAX    = 0.20              # table: only show hz <= this (h_z^c = 0.1939)
HZ_PLOT = [0.1, 0.15]       # §3: which hz slices to draw learning curves for

COL = dict(zip(LS, plt.cm.plasma(np.linspace(0, 0.85, len(LS)))))

DATA = {}
for L in LS:
    f = f"{ENERGY_DIR}/energy_L{L}_hx0.0.json"
    if os.path.exists(f):
        DATA[L] = json.load(open(f))
print("loaded L =", sorted(DATA))

In [ ]:
# ====================== 2 · E_NQS vs exact perturbation series ======================
for L in sorted(DATA):
    d = DATA[L]
    h  = np.array(d["field"], float)
    E  = np.array(d["E"], float)
    ee = np.array(d["E_spread"], float)          # MC error of the mean
    c  = counts(L, BC)
    m  = h <= HMAX

    print(f"\nL={L}  N={c.N}  E0={c.E0:.0f}  c2={c.c_z2:.2f}  c4={c.c_z4:.2f}   (c2, c4 exact)")
    print(f"  {'hz':>6} {'E_NQS':>12} {'E_err':>8} {'E_pert2':>12} {'E_pert4':>12} "
          f"{'rel_err2':>9} {'rel_err4':>9} {'sigma':>7} {'c6~':>8}")
    for hi, Ei, ei in zip(h[m], E[m], ee[m]):
        p2 = E_lowfield(L, BC, hz=hi)
        p4 = E_lowfield(L, BC, hz=hi, order=4)
        print(f"  {hi:6.3f} {Ei:12.4f} {ei:8.5f} {p2:12.4f} {p4:12.4f} "
              f"{abs(Ei-p2)/abs(Ei):9.2e} {abs(Ei-p4)/abs(Ei):9.2e} "
              f"{(Ei-p4)/ei:+7.2f} {0.45*c.N*hi**6:8.1e}")

## 3 · Learning curves against the exact value

**Left:** energy *per site* (so all $L$ share a scale) with the exact $O(h_z^4)$ target as a
dashed line of the matching colour. **Right:** the same approach as a relative residual on a log
axis, with a dotted line per $L$ at the MC error floor
($\texttt{energy\_err}/|E_{\rm pert}|$ at the last step) — below that a curve cannot meaningfully
descend, so a curve *oscillating around* its floor has converged as far as is measurable, while a
curve on a **flat plateau above** its floor has stopped descending for another reason.

Curves are read from the per-run dumps in the **gitignored** `data/tc_nqs/` mirror, not from the
aggregated JSONs.

In [ ]:
# ====================== 3 · LEARNING CURVES vs the exact perturbation energy ==========
# Curves live in the per-run dumps (gitignored mirror), NOT in the aggregates.
def load_curve(L, hz):
    hits = glob.glob(f"{CURVE_DIR}/L{L}/*_L{L}_hx0.0_hz{hz}.json")
    hits = [h for h in hits if not h.endswith(".curve.json")]
    if not hits:
        return None
    d = json.load(open(hits[0]))
    return d["curve"], d["observables"]

nrow = len(HZ_PLOT)
fig, ax = plt.subplots(nrow, 2, figsize=(11, 4.0 * nrow), squeeze=False)
missing = []
for r, hz in enumerate(HZ_PLOT):
    aL, aR = ax[r, 0], ax[r, 1]
    for L in sorted(DATA):
        got = load_curve(L, hz)
        if got is None:
            missing.append((L, hz)); continue
        cur, _ = got
        c    = counts(L, BC)
        step = np.array(cur["step"], float)
        Ec   = np.array(cur["energy"], float)
        Ece  = np.array(cur["energy_err"], float)
        Ep   = E_lowfield(L, BC, hz=hz, order=4)          # exact through O(hz^4)

        # LEFT: energy per site (comparable across L) + the theoretical target
        aL.plot(step, Ec / c.N, "-", lw=1.2, color=COL[L], label=f"L={L}")
        aL.axhline(Ep / c.N, ls="--", lw=1, color=COL[L], alpha=0.7)

        # RIGHT: approach to the target, and the MC noise floor it cannot go below
        aR.semilogy(step, np.abs(Ec - Ep) / abs(Ep), "-", lw=1.2, color=COL[L], label=f"L={L}")
        aR.axhline(Ece[-1] / abs(Ep), ls=":", lw=1, color=COL[L], alpha=0.8)
        # The final `observables.E0` is an INDEPENDENT fresh vs.expect, not the last
        # curve point -- at the noise floor the two differ by up to ~16x (L=4). Drawn
        # offset to the right of the curve, with its own MC error bar, so it reads as a
        # separate summary estimate rather than a stray point on the curve.

    aL.set(xlabel="step", ylabel=r"$E/N$", title=rf"$h_z={hz}$ — energy per site")
    lo = min(E_lowfield(L, BC, hz=hz, order=4) / counts(L, BC).N for L in DATA)
    aL.set_ylim(lo - 0.06, lo + 0.20)
    aR.set(xlabel="step", ylabel=r"$|E-E_{\rm pert}|/|E_{\rm pert}|$",
           title=rf"$h_z={hz}$ — approach to the exact $O(h_z^4)$ value")
    if r == 0:
        aL.legend(fontsize=8, loc="best")
        aR.legend(fontsize=8, loc="best")

fig.suptitle(r"Learning curves vs exact perturbation energy ($h_x=0$).  dashed $=E_{\rm pert}$, "
             r"dotted $=$ MC error floor", y=1.005)
fig.tight_layout(); plt.show()
fig.savefig(f"{FIGDIR}/energy_benchmarks_learning_curves.png", dpi=300, bbox_inches="tight")
if missing:
    print("no curve found for (L, hz):", missing,
          f"\n  curves come from the gitignored mirror {CURVE_DIR}")

## 4 · QMC arm — the mixed point $(h_x, h_z) = (0.2, 0.1)$

On the mixed cut no exact $c_4$ exists, so the series alone cannot separate its own truncation
from the NQS variational error. The arbiter is continuous-time QMC (**ParaToric**, Linsel &
Pollet, SciPost Phys. Codebases 75 (2026); Wu et al., PRB 85, 195104 (2012)), run on this
machine and validated against every exact handle we have: $h=0$ anchors at $L=2,4,5,6,7$
(geometry certificate), $L=2$ ED, the pure-$h_z$ 4th-order series, and $\beta$-doubling
($\beta=12$, thermal error $<10^{-7}$).

Since QMC estimates $E_0$ without variational bias, $\varepsilon = E_{\rm NQS} - E_{\rm QMC}$
**is** the NQS variational error (necessarily $\geq 0$), measured with
$\sigma = \sqrt{\sigma_{\rm QMC}^2 + \sigma_{\rm NQS}^2}$. The figure shows both energies
relative to the exact 2nd-order series: the QMC$-$series offset is the true series truncation,
the NQS$-$QMC gap is $\varepsilon$.

In [ ]:
# ====================== 4 · QMC arm: mixed point (hx, hz) = (0.2, 0.1) ======================
QMC_HX, QMC_HZ = 0.2, 0.1
QMC_DIR = f"{ROOT}/qmc_hx{QMC_HX}_hz{QMC_HZ}"

rows = []
for L in LS:
    fq, fn = f"{QMC_DIR}/paratoric_L{L}.json", f"{ROOT}/phase_hx{QMC_HX}_energy/energy_L{L}_hx{QMC_HX}.json"
    if not (os.path.exists(fq) and os.path.exists(fn)):
        continue
    q, d = json.load(open(fq)), json.load(open(fn))
    i = d["field"].index(QMC_HZ)
    En, en = d["E"][i], d["E_spread"][i]
    ser2 = E_lowfield(L, BC, hx=QMC_HX, hz=QMC_HZ)            # exact through O(h^2)
    eps, sig = En - q["E"], float(np.hypot(en, q["E_err"]))
    rows.append(dict(L=L, Eq=q["E"], eq=q["E_err"], En=En, en=en,
                     ser2=ser2, eps=eps, sig=sig, z=eps / sig))

print(f"  {'L':>2} {'E_QMC':>12} {'err':>7} {'E_NQS':>12} {'err':>7} "
      f"{'eps=E_NQS-E_QMC':>16} {'z':>5}  {'series2-E_QMC':>13}")
for r in rows:
    print(f"  {r['L']:2d} {r['Eq']:12.4f} {r['eq']:7.4f} {r['En']:12.4f} {r['en']:7.4f} "
          f"{r['eps']:+10.4f} +- {r['sig']:.4f} {r['z']:5.1f}  {r['ser2']-r['Eq']:13.4f}")

fig, ax = plt.subplots(figsize=(5.4, 3.8))
for r in rows:
    ax.errorbar(r["L"] - 0.05, r["Eq"] - r["ser2"], yerr=r["eq"], fmt="o",
                color=COL[r["L"]], ms=7, capsize=3, zorder=3)
    ax.errorbar(r["L"] + 0.05, r["En"] - r["ser2"], yerr=r["en"], fmt="s",
                mfc="white", color=COL[r["L"]], ms=7, capsize=3, zorder=3)
ax.axhline(0.0, color="0.5", lw=0.9, ls="--")
ax.errorbar([], [], fmt="o", color="0.3", label=r"$E_{\rm QMC}$ (ParaToric)")
ax.errorbar([], [], fmt="s", mfc="white", color="0.3", label=r"$E_{\rm NQS}$")
ax.plot([], [], ls="--", color="0.5", label=r"exact 2nd-order series")
ax.set_xticks([r["L"] for r in rows])
ax.set_xlabel(r"$L$")
ax.set_ylabel(r"$E - E^{(2)}_{\rm series}$")
ax.set_title(rf"$(h_x, h_z) = ({QMC_HX}, {QMC_HZ})$: NQS$-$QMC gap $=$ variational error",
             fontsize=10)
ax.legend(frameon=False, fontsize=9, loc="upper left")
fig.tight_layout()
fig.savefig(f"{FIGDIR}/energy_benchmarks_qmc_mixed.png", dpi=300, bbox_inches="tight")
plt.show()

## 5 · Learning curves against the QMC benchmark — mixed point

Same construction as §3, but on the mixed cut the target line is $E_{\rm QMC}$ (ParaToric,
§4) instead of a perturbation series — the only non-variational reference that exists here.
**Left:** energy per site with the QMC value as the dashed target. **Right:** relative distance
to the QMC value on a log axis; dotted line $=$ each run's own MC error floor, and the grey
band marks $\sigma_{\rm QMC}/|E_{\rm QMC}|$ — inside it the benchmark itself cannot
distinguish. A curve flattening *above* both floors has a resolved variational error; the gap
it converges to is the $\varepsilon$ of the §4 table.

In [ ]:
# ====================== 5 · LEARNING CURVES vs the QMC benchmark ======================
QMC_CURVE_DIR = os.path.join(os.path.dirname(ROOT), f"data/tc_nqs/phase_hx{QMC_HX}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
aL, aR = ax
missing = []
for r in rows:                                     # rows from §4: one entry per L with QMC data
    L = r["L"]
    hits = [h for h in glob.glob(f"{QMC_CURVE_DIR}/L{L}/*_L{L}_hx{QMC_HX}_hz{QMC_HZ}.json")
            if not h.endswith(".curve.json")]
    if not hits:
        missing.append(L); continue
    cur = json.load(open(hits[0]))["curve"]
    c    = counts(L, BC)
    step = np.array(cur["step"], float)
    m150 = step <= 150                       # common x-window (L=7 was resumed to 175)
    step, Ec, Ece = step[m150], np.array(cur["energy"], float)[m150], \
                    np.array(cur["energy_err"], float)[m150]
    Eq, eq = r["Eq"], r["eq"]

    aL.plot(step, Ec / c.N, "-", lw=1.2, color=COL[L], label=f"L={L}")
    aL.axhline(Eq / c.N, ls="--", lw=1, color=COL[L], alpha=0.7)

    aR.semilogy(step, np.abs(Ec - Eq) / abs(Eq), "-", lw=1.2, color=COL[L], label=f"L={L}")
    aR.axhline(Ece[-1] / abs(Eq), ls=":", lw=1, color=COL[L], alpha=0.8)

aL.set(xlabel="step", ylabel=r"$E/N$",
       title=rf"$(h_x,h_z)=({QMC_HX},{QMC_HZ})$ — energy per site")
lo = min(r["Eq"] / counts(r["L"], BC).N for r in rows)
aL.set_ylim(lo - 0.06, lo + 0.20)
aL.legend(fontsize=8, loc="upper right")

aR.set(xlabel="step", ylabel=r"$|E-E_{\rm QMC}|/|E_{\rm QMC}|$",
       title=rf"$(h_x,h_z)=({QMC_HX},{QMC_HZ})$ — approach to the QMC benchmark")
aR.legend(fontsize=8, loc="best")
# fig.suptitle(r"Learning curves vs QMC (ParaToric).  dashed $=E_{\rm QMC}$, "
#              r"dotted $=$ MC error floor, grey band $=$ QMC resolution", y=1.005)
fig.tight_layout(); plt.show()
fig.savefig(f"{FIGDIR}/energy_benchmarks_learning_curves_qmc.png", dpi=300, bbox_inches="tight")
if missing:
    print("no curve found for L:", missing, f"\n  curves come from the gitignored mirror {QMC_CURVE_DIR}")

## 6 · Architecture comparison against the QMC baseline — $(h_x, h_z) = (0.2, 0.2)$, $L=4$

Symmetry-aware (`gridinv`) vs symmetry-unaware (GeoCNN) learning curves (3 seeds each, W&B
export) against the ParaToric baseline $E_{\rm QMC} = -174.596(15)$ — itself certified at this
near-boundary point by an $L=2$ ED check, $\chi^2$-clean decorrelation (4$\times$), and
$\beta$-independence across $\beta = 12/24/48$. The grey band is the QMC resolution
$\sigma_{\rm QMC}/|E_{\rm QMC}|$: converging "to QMC" means entering this band.

In [ ]:
# ====================== 6 · ARCHITECTURE COMPARISON vs QMC baseline ======================
import csv

Q = json.load(open(f"{ROOT}/qmc_hx0.2_hz0.2/paratoric_L4_combined.json"))
Eq, eq, Nq = Q["E"], Q["E_err"], counts(4, BC).N

with open(f"{ROOT}/arch_compare/wandb_arch_bench_hx0.2_hz0.2_L4.csv") as f:
    rdr = list(csv.reader(f))
hdr = rdr[0]
dat = np.array([[float(x) if x else np.nan for x in row] for row in rdr[1:]])
step = dat[:, 0]
runs = {h[:-len(" - energy")]: dat[:, j] for j, h in enumerate(hdr) if h.endswith(" - energy")}
groups = {"Approx. NQS":  {k: v for k, v in runs.items() if k.startswith("bench_aware")},
          "CNN": {k: v for k, v in runs.items() if k.startswith("bench_unaware")}}
# plasma-derived group colors (house palette); seeds = lightness shades of the base
import matplotlib.colors as mcolors
GBASE = {"Approx. NQS": plt.cm.plasma(0.05), "CNN": plt.cm.plasma(0.80)}
def _shades(base, n):                       # lightest -> full base colour
    b = np.array(mcolors.to_rgb(base))
    return [tuple(1 - f * (1 - b)) for f in np.linspace(0.45, 1.0, max(2, n))]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
aL, aR = ax
for gname, g in groups.items():
    shades = _shades(GBASE[gname], len(g))
    for i, (k, E) in enumerate(sorted(g.items())):
        m = np.isfinite(E)
        lbl = gname if i == len(g) - 1 else None      # legend keyed to darkest shade
        aL.plot(step[m], E[m] / Nq, "-", lw=1.2, color=shades[i], label=lbl)
        aR.semilogy(step[m], np.abs(E[m] - Eq) / abs(Eq), "-", lw=1.2,
                    color=shades[i], label=lbl)
    fin = [E[np.isfinite(E)][-1] for E in g.values()]
    print(f"{gname:28s} final E = {np.mean(fin):10.3f} (seeds: "
          + ", ".join(f"{v:.3f}" for v in sorted(fin)) + f")   gap to QMC = {np.mean(fin)-Eq:+.3f}")

aL.axhline(Eq / Nq, ls="--", lw=1.2, color="0.25", label=r"QMC")
aL.set(xlabel="step", ylabel=r"$E/N$",
       title=r"$(h_x,h_z)=(0.2,0.2)$ — energy per site")
aL.set_ylim(Eq / Nq - 0.02, Eq / Nq + 0.10)
aL.legend(fontsize=8, loc="upper right")

# inset: end of training (steps 150-200) -- seed-to-seed spread of the two architectures
axins = aL.inset_axes([0.30, 0.40, 0.42, 0.38])
w = step >= 150
for gname, g in groups.items():
    shades = _shades(GBASE[gname], len(g))
    for i, (k, E) in enumerate(sorted(g.items())):
        m = np.isfinite(E) & w
        axins.plot(step[m], E[m] / Nq, "-", lw=1.0, color=shades[i])
vals = np.concatenate([E[np.isfinite(E) & w] / Nq for g in groups.values() for E in g.values()])
pad = 0.10 * (vals.max() - vals.min())
axins.set_xlim(150, step[np.isfinite(step)].max())
axins.set_ylim(min(vals.min(), (Eq - eq) / Nq) - pad, vals.max() + pad)
axins.axhline(Eq / Nq, ls="--", lw=1, color="0.25")   # QMC baseline inside the zoom
axins.set_xticklabels([]); axins.set_yticklabels([])
for sp in axins.spines.values():                    # full black box around the inset
    sp.set_visible(True); sp.set_color("black")
_rect, _conn = aL.indicate_inset_zoom(axins, edgecolor="black")
for cl in _conn:                                    # connectors default to 50% alpha
    cl.set_alpha(1.0); cl.set_color("black")
_rect.set_alpha(1.0); _rect.set_edgecolor("black")  # source rectangle too
aR.set(xlabel="step", ylabel=r"$|E-E_{\rm QMC}|/|E_{\rm QMC}|$",
       title=r"$(h_x,h_z)=(0.2,0.2)$ — approach to the QMC baseline")
aR.legend(fontsize=8, loc="upper right")
# fig.suptitle(rf"Symmetry-aware vs -unaware NQS at $(h_x,h_z)=(0.2,0.2)$, $L=4$.  "
#              rf"$E_{{\rm QMC}}={Eq:.3f}({eq*1000:.0f})$, grey $=$ QMC resolution", y=1.005)
fig.tight_layout(); plt.show()
fig.savefig(f"{FIGDIR}/arch_compare_qmc_L4_hx0.2_hz0.2.png", dpi=300, bbox_inches="tight")

## 7 · Combined report figure — architecture comparison + system-size scaling

Left: CNN vs Approx. NQS at $(h_x,h_z)=(0.2,0.2)$, $L=4$ (§6). Right: Approx. NQS learning
curves for $L=4\ldots7$ at $(h_x,h_z)=(0.2,0.1)$ against their per-$L$ QMC values (§5).
Energies per site throughout; dashed $=E_{\rm QMC}$.

In [ ]:
# ====================== 7 · COMBINED: comparison (left) + scaling (right) ======================
fig, (aC, aS) = plt.subplots(1, 2, figsize=(11, 4.2))

# --- left: architecture comparison (reuses §6 data: groups, GBASE, _shades, Eq, eq, Nq) ---
for gname, g in groups.items():
    shades = _shades(GBASE[gname], len(g))
    for i, (k, E) in enumerate(sorted(g.items())):
        m = np.isfinite(E)
        aC.plot(step[m], E[m] / Nq, "-", lw=1.2, color=shades[i],
                label=gname if i == len(g) - 1 else None)
aC.axhline(Eq / Nq, ls="--", lw=1.2, color="0.25", label=r"QMC")
aC.set(xlabel="step", ylabel=r"$E/N$",
       title=r"$a) (h_x,h_z)=(0.2,0.2)$ — architectures, $L=4$")
aC.set_ylim(Eq / Nq - 0.02, Eq / Nq + 0.10)
aC.legend(fontsize=8, loc="upper right")

axins = aC.inset_axes([0.30, 0.40, 0.42, 0.38])
w = step >= 150
for gname, g in groups.items():
    shades = _shades(GBASE[gname], len(g))
    for i, (k, E) in enumerate(sorted(g.items())):
        m = np.isfinite(E) & w
        axins.plot(step[m], E[m] / Nq, "-", lw=1.0, color=shades[i])
axins.axhline(Eq / Nq, ls="--", lw=1, color="0.25")
vals = np.concatenate([E[np.isfinite(E) & w] / Nq for g in groups.values() for E in g.values()])
pad = 0.10 * (vals.max() - vals.min())
axins.set_xlim(150, step[np.isfinite(step)].max())
axins.set_ylim(min(vals.min(), Eq / Nq) - pad, vals.max() + pad)
axins.set_xticklabels([]); axins.set_yticklabels([])
for sp in axins.spines.values():
    sp.set_visible(True); sp.set_color("black")
_rect, _conn = aC.indicate_inset_zoom(axins, edgecolor="black")
for cl in _conn:
    cl.set_alpha(1.0); cl.set_color("black")
_rect.set_alpha(1.0); _rect.set_edgecolor("black")

# --- right: system-size scaling (reuses §4 rows + §5 loader pattern; <=150 steps) ---
for r in rows:
    L = r["L"]
    hits = [h for h in glob.glob(f"{QMC_CURVE_DIR}/L{L}/*_L{L}_hx{QMC_HX}_hz{QMC_HZ}.json")
            if not h.endswith(".curve.json")]
    if not hits:
        continue
    cur = json.load(open(hits[0]))["curve"]
    c = counts(L, BC)
    st = np.array(cur["step"], float)
    m = st <= 150
    aS.plot(st[m], np.array(cur["energy"], float)[m] / c.N, "-", lw=1.2,
            color=COL[L], label=f"L={L}")
    aS.axhline(r["Eq"] / c.N, ls="--", lw=1, color=COL[L], alpha=0.7)
aS.set(xlabel="step", ylabel=r"$E/N$",
       title=r"$b) (h_x,h_z)=(0.2,0.1)$ — system sizes")
lo = min(r["Eq"] / counts(r["L"], BC).N for r in rows)
aS.set_ylim(lo - 0.015, lo + 0.20)
aS.legend(fontsize=8, loc="upper right")

fig.tight_layout(); plt.show()
fig.savefig(f"{FIGDIR}/combined_arch_and_scaling.png", dpi=300, bbox_inches="tight")